In [1]:
# compile all multi-omics data for dysregulated protein targets for IPA enriched and shared regulators
import pandas as pd

In [ ]:
# import MRSA and PAO1 DE data
DE_res_6hr_MRSA = pd.read_excel('../data/DEG_RNA/DEresults_6hrhr.xlsx', sheet_name='res_MRSA')
DE_res_24hr_MRSA = pd.read_excel('../data/DEG_RNA/DEresults_24hrhr.xlsx', sheet_name='res_MRSA')


In [ ]:
prot_res_ANOVA = pd.read_csv('../data/PROT_ANOVA.csv', index_col=0)


In [4]:
xls = pd.ExcelFile('../results/IPA_downstream_analysis/PROT_sharedreg.xlsx')
xls.sheet_names

['CUL4B', 'Pkg', 'Rac', 'ROCK', 'ATM', 'AREG', 'MYC']

In [ ]:
import pickle
with open('../results/IPA_downstream_analysis/prot_name_d.obj', 'rb') as input_file:
    prot_name_d = pickle.load(input_file)


In [25]:
sheet='MYC'
data = pd.read_excel(xls, sheet_name=sheet, index_col=0)
data.head()

,1hr_z-2.673_symbol,1hr_z-2.673_reg,1hr_z-2.673_uniprot,2hr_z-2.412_symbol,2hr_z-2.412_reg,2hr_z-2.412_uniprot,47hr_z-2.714_symbol,47hr_z-2.714_reg,47hr_z-2.714_uniprot,24hr_z-3.795_symbol,24hr_z-3.795_reg,24hr_z-3.795_uniprot
0,AKT1,0.0,P31749,AKT1,0.0,P31749,ABCC1,0,P33527,ABCC1,0.0,P33527
1,BAG1,0.0,J3QTA2,BAG1,0.0,J3QTA2,AKT1,0,P31749,ALCAM,0.0,Q13740
2,EIF4E,0.0,P06730,BAX,0.0,Q07812,ALCAM,0,Q13740,BAG1,0.0,J3QTA2
3,IFIT2,0.0,A0A087X279,BCAT1,0.0,P54687,ALDH18A1,0,P54886,BAX,0.0,Q07812
4,IQGAP2,0.0,Q13576,BCOR,0.0,Q6W2J9,BAG1,0,J3QTA2,BCOR,0.0,Q6W2J9


In [ ]:
reg_data = [{(i.split('_')[0],i.split('_')[1]):[list(data[i].dropna()), list(data[i.split('_')[0]+'_'+i.split('_')[1]+'_'+'uniprot'].dropna())]} \
            for i in data.columns if 'symbol' in i]

reg_d = {}

for d in reg_data:
    k = list(d.keys())[0]
    v = d[k]
    df = pd.DataFrame(columns=['GENE', 'UNIPROT', 
                               'DEG_6HR_MRSA_LOG2FC', 'DEG_6HR_MRSA_ADJPVAL', 'DEG_24HR_MRSA_LOG2FC', 'DEG_24HR_MRSA_ADJPVAL', 
                               'PROT_UNIF_1H_LOG2FC', 'PROT_UNIF_2H_LOG2FC','PROT_UNIF_4/7H_LOG2FC', 'PROT_UNIF_24H_LOG2FC', 'PROT_ADJPVAL') 
    for i in range(len(v[0])):
        gene = v[0][i]
        uniprot = v[1][i]
        
        try:
            deg_6hr_mrsa_fc = list(DE_res_6hr_MRSA.loc[DE_res_6hr_MRSA['hgnc_symbol']==gene]['log2FoldChange'])[0] 
            deg_6hr_mrsa_pval = list(DE_res_6hr_MRSA.loc[DE_res_6hr_MRSA['hgnc_symbol']==gene]['padj'])[0]
        except:
            deg_6hr_mrsa_fc='nan'
            deg_6hr_mrsa_pval='nan'
            
        try:
            deg_24hr_mrsa_fc = list(DE_res_24hr_MRSA.loc[DE_res_24hr_MRSA['hgnc_symbol']==gene]['log2FoldChange'])[0] 
            deg_24hr_mrsa_pval = list(DE_res_24hr_MRSA.loc[DE_res_24hr_MRSA['hgnc_symbol']==gene]['padj'])[0]
        except:
            deg_24hr_mrsa_fc='nan'
            deg_24hr_mrsa_pval='nan'
            
        
        
        # uniprot id with , creates a multiple mapping problem 
        # need to seperate by , and check for uniprot id match
        try:
            prot_1hr_fc=prot_res_ANOVA.loc[uniprot]['UNIF_1H_LOG2FC']
            prot_2hr_fc=prot_res_ANOVA.loc[uniprot]['UNIF_2H_LOG2FC']
            prot_47hr_fc=prot_res_ANOVA.loc[uniprot]['UNIF_4/7H_LOG2FC']
            prot_24hr_fc=prot_res_ANOVA.loc[uniprot]['UNIF_24H_LOG2FC']
            prot_anova_pval=prot_res_ANOVA.loc[uniprot]['ADJPVAL']
        except:
            try:
                new_uniprot=prot_name_d[gene.upper()]
                prot_1hr_fc=prot_res_ANOVA.loc[new_uniprot]['UNIF_1H_LOG2FC']
                prot_2hr_fc=prot_res_ANOVA.loc[new_uniprot]['UNIF_2H_LOG2FC']
                prot_47hr_fc=prot_res_ANOVA.loc[new_uniprot]['UNIF_4/7H_LOG2FC']
                prot_24hr_fc=prot_res_ANOVA.loc[new_uniprot]['UNIF_24H_LOG2FC']
                prot_anova_pval=prot_res_ANOVA.loc[new_uniprot]['ADJPVAL']
            except:
                prot_1hr_fc='nan'
                prot_2hr_fc='nan'
                prot_47hr_fc='nan'
                prot_24hr_fc='nan'
                prot_anova_pval='nan'
        
        
        items = [gene, uniprot, 
                 deg_6hr_mrsa_fc, deg_6hr_mrsa_pval, deg_24hr_mrsa_fc, deg_24hr_mrsa_pval,
                 prot_1hr_fc, prot_2hr_fc, prot_47hr_fc, prot_24hr_fc, prot_anova_pval]
        df.loc[len(df.index)] = items
    reg_d[k]=df

In [27]:
writer = pd.ExcelWriter('../results/IPA_downstream_analysis/regulator_pathways/{}_REGULATOR_PATHWAYS.xlsx'.format(sheet), engine='openpyxl')
for df_name, df in reg_d.items():
    df.to_excel(writer, sheet_name='_'.join(df_name))
writer.save()